# 02 — Nettoyage et modélisation en étoile

**Objectif :** appliquer les décisions de l'audit (`01_data_audit.ipynb`), construire le modèle en étoile et vérifier le résultat avant chargement dans SQL Server.

Toute la logique vit dans **`src/etl.py`** (source unique, testable, rejouable avec `python -m src.etl`). Ce notebook l'exécute étape par étape et **contrôle** chaque sortie.

```
                 Dim_Customer
                      │
Dim_Date ──────── Fact_Sales ──────── Dim_Product
                      │
                 Dim_Store ── Dim_Employee

Dim_Discount : référence autonome des périodes promotionnelles
Dim_Currency : taux de change (créée en SQL)
```

In [1]:
import sys
sys.path.append("..")  # rend le package src/ importable depuis notebooks/

import pandas as pd

from src import etl

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)

In [2]:
raw = etl.load_raw_data()
clean = etl.clean_all(raw)

## 1. Contrôles du nettoyage

In [3]:
pd.DataFrame({
    "Lignes avant": {name: len(df) for name, df in raw.items()},
    "Lignes après": {name: len(df) for name, df in clean.items()},
    "Manquants après": {name: int(df.isna().sum().sum()) for name, df in clean.items()},
})

,Lignes avant,Lignes après,Manquants après
customers,1643306,1643306,0
products,17940,17940,0
discounts,181,181,0
employees,404,404,0
stores,35,35,0
transactions,6416827,6416029,0


In [4]:
tx = clean["transactions"]
print("Doublons exacts restants            :", tx.duplicated().sum())
print("Lignes financièrement incohérentes  :", etl.count_financial_inconsistencies(tx), "(avant :",
      etl.count_financial_inconsistencies(raw["transactions"].drop_duplicates()), ")")
tx.groupby("Transaction Type")["Discount"].describe()

Doublons exacts restants            : 0


Lignes financièrement incohérentes  : 0 (avant : 97931 )


,count,mean,std,min,25%,50%,75%,max
Transaction Type,,,,,,,,
Return,"338,829.00",0.12,0.20,0.00,0.00,0.00,0.25,0.60
Sale,"6,077,200.00",0.13,0.20,0.00,0.00,0.00,0.25,0.60


In [5]:
clean["stores"][["Country", "City", "Store Name"]].drop_duplicates("Country")

,Country,City,Store Name
0,United States,New York,Store New York
5,China,Shanghai,Store Shanghai
10,Germany,Berlin,Store Berlin
15,United Kingdom,London,Store London
20,France,Paris,Store Paris
25,Spain,Madrid,Store Madrid
30,Portugal,Lisboa,Store Lisboa


Seules les 798 lignes dupliquées ont été retirées, plus aucune valeur manquante, et la règle `Line Total = ±Prix x Quantité x (1 - Remise)` est désormais vérifiée sur toutes les lignes (retours compris).

## 2. Construction du modèle en étoile

Choix de modélisation :
- **Grain de `Fact_Sales`** : une ligne de facture. Ventes et retours sont dans la même table (montant négatif pour un retour), ce qui permet de calculer CA brut, retours et CA net avec une seule source.
- **`Date_Key` au format AAAAMMJJ** : lisible et triable ; `Dim_Date` couvre chaque jour de la période, y compris les jours sans vente.
- **Âge client calculé à la dernière date du dataset**, et non à la date d'exécution : le résultat ne change pas selon le jour où l'on relance le pipeline.
- **`Dim_Discount` non reliée à `Fact_Sales`** : la remise effectivement appliquée est déjà sur chaque ligne ; la table sert à analyser le calendrier promotionnel.

In [6]:
tables = etl.build_star_schema(clean)
pd.Series({name: df.shape for name, df in tables.items()}, name="Dimensions")

dim_date            (808, 11)
dim_customer    (1643306, 10)
dim_product        (17940, 7)
dim_store             (35, 8)
dim_employee         (404, 4)
dim_discount         (181, 7)
fact_sales      (6416029, 18)
Name: Dimensions, dtype: object

In [7]:
tables["fact_sales"].head()

,Invoice_ID,Line,Date_Key,Customer_Key,Product_Key,Store_Key,Employee_Key,SKU,Size,Color,Quantity,Unit_Price,Discount,Line_Total,Invoice_Total,Transaction_Type,Payment_Method,Currency
0,INV-US-001-03558761,1,20230101,47162,485,1,7,MASU485-M-,M,Unknown,1,80.50,0.00,80.50,126.70,Sale,Cash,USD
1,INV-US-001-03558761,2,20230101,47162,2779,1,7,CHCO2779-G-,G,Unknown,1,31.50,0.40,18.90,126.70,Sale,Cash,USD
2,INV-US-001-03558761,3,20230101,47162,64,1,7,MACO64-M-NEUTRAL,M,NEUTRAL,1,45.50,0.40,27.30,126.70,Sale,Cash,USD
3,INV-US-001-03558762,1,20230101,10142,131,1,6,FECO131-M-BLUE,M,BLUE,1,70.00,0.40,42.00,77.00,Sale,Cash,USD
4,INV-US-001-03558762,2,20230101,10142,716,1,6,MAT-716-L-WHITE,L,WHITE,1,26.00,0.00,26.00,77.00,Sale,Cash,USD


In [8]:
tables["dim_date"].head()

,Date_Key,Date,Year,Quarter,Month,Month_Name,Day,Day_Of_Week,Day_Name,Is_Weekend,Week_Of_Year
0,20230101,2023-01-01,2023,1,1,January,1,7,Sunday,True,52
1,20230102,2023-01-02,2023,1,1,January,2,1,Monday,False,1
2,20230103,2023-01-03,2023,1,1,January,3,2,Tuesday,False,1
3,20230104,2023-01-04,2023,1,1,January,4,3,Wednesday,False,1
4,20230105,2023-01-05,2023,1,1,January,5,4,Thursday,False,1


## 3. Validation avant chargement

In [9]:
etl.validate_star_schema(tables)  # lève une erreur si une clé est orpheline ou dupliquée
print("Modèle en étoile valide : clés primaires uniques, aucune clé étrangère orpheline.")

fact_sales : 21514 doublons sur (Invoice_ID, Line)


Modèle en étoile valide : clés primaires uniques, aucune clé étrangère orpheline.


In [10]:
tables["dim_customer"]["Age"].describe()

count   1,643,306.00
mean           31.41
std            11.82
min            18.00
25%            21.00
50%            28.00
75%            39.00
max            75.00
Name: Age, dtype: float64

## 4. Export vers `data/processed/`

In [11]:
etl.export_tables(tables)

**Étape suivante :** créer la base (`sql/01` -> `sql/03`) puis charger ces fichiers avec `python -m src.load_to_sql_server`. Les analyses suivantes (notebooks 03 à 06) interrogent directement SQL Server, via la vue `warehouse.vw_Fact_Sales_USD`.